<a href="https://colab.research.google.com/github/Alisherbakh/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "trl==1.14.2" transformers torch bitsandbytes datasets peft #trl

In [2]:
import trl
print(trl.__version__)    # → 1.14.2 bo'lishi kerak

1.14.2


In [3]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model                   # PEFT kutubxonasidan LoRA sozlamalari klassini chaqiramiz
from trl import SFTTrainer


In [4]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'      # Hugging Face'dagi model nomi: "muallif/model_nomi"
tokenizer = AutoTokenizer.from_pretrained(model_id)  # Shu modelga mos tayyor tokenizerni (lug'at + bo'lish qoidalari) yuklab olamiz

#print('Vocab size:', tokenizer.vocab_size)               # Lug'atdagi tokenlar soni → 32000
#print('Special tokens:', tokenizer.special_tokens_map)   # Maxsus tokenlar → {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}


# quantization
bnb_config = BitsAndBytesConfig(                  # Modelni siqish (quantization) sozlamalari
    load_in_4bit=True,                            # Og'irliklarni 4 bitga siqib yuklaymiz (odatda 16 bit bo'ladi)
    bnb_4bit_use_double_quant=True,               # Siqish uchun kerak bo'lgan yordamchi sonlarni ham siqamiz → yana ozgina xotira tejaladi
    bnb_4bit_compute_dtype=torch.bfloat16,        # Hisoblash paytida sonlar vaqtincha bfloat16 ga ochiladi (aniqlik uchun)
    bnb_4bit_quant_type='nf4'                     # Siqish usuli: NF4 — neyron tarmoq og'irliklari uchun maxsus 4 bitli format
)

bnb_config                                        # Sozlamalarni ekranga chiqaramiz (tekshirish uchun)

model = AutoModelForCausalLM.from_pretrained(     # Matn yozuvchi (keyingi tokenni bashorat qiluvchi) tayyor modelni yuklaymiz
    model_id,                                     # Qaysi model (tokenizer bilan bir xil bo'lishi shart)
    quantization_config=bnb_config,               # Yuqoridagi 4 bitli siqish sozlamalarini qo'llaymiz
    device_map='auto',   # GPU, CPU                 → model qismlarini GPU va CPU'ga avtomatik taqsimlaydi
    dtype=torch.bfloat16                          # Siqilmagan qismlar uchun ma'lumot turi
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# Before Fine-tuning                                   → modelni o'zimizning ma'lumotda o'qitishdan OLDIN qanday javob berishini ko'ramiz
prompt = "Explain what a tokenizer is?"                # Modelga beriladigan savol (prompt)
# prompt = "A tokenizer is a tool in natural language processing that"   → muqobil: gapni boshlab berib, davomini yozdirish

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)   # GPU, CPU   → promptni tensorga aylantirib, model turgan qurilmaga ko'chiramiz

with torch.no_grad():                                  # Gradient hisoblamaymiz (faqat javob olyapmiz, o'qitmayapmiz)
    output_ids = model.generate(                       # Model javob matnini token-token yozadi
        **inputs,                                      # input_ids va attention_mask ni argument qilib beramiz
        max_new_tokens=80,                             # Ko'pi bilan 80 ta YANGI token yozsin
        do_sample=True,                                # Har safar eng ehtimollisini emas, ehtimollik bo'yicha tasodifiy tanlaydi
        temperature=0.7                                # Tasodifiylik darajasi: past → aniqroq, baland → ijodkorroq
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))   # Javob ID'larini matnga aylantirib chiqaramiz (<s>, </s> siz)

# print(model)                                  → butun modelni chiqarish o'chirib qo'yildi (juda uzun)
first_block = model.model.layers[0]             # 22 ta Transformer blokdan birinchisini (0-indeks) olamiz
print('first_block:', first_block)              # Shu bitta blokning ichki tuzilmasini chiqaramiz
print('=======')                                # Ajratuvchi chiziq
print(first_block.self_attn)                    # Blok ichidagi Attention (diqqat) qismini alohida chiqaramiz
print('=======')                                # Ajratuvchi chiziq
print(model.config)                             # Modelning barcha sozlamalarini (o'lchamlar, qatlamlar soni) chiqaramiz

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is? Provide an example and how it is used in a machine learning model.
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features

In [6]:
def count_parameters(model):                                  # Modeldagi parametrlar sonini sanaydigan funksiya
    return sum(p.numel() for p in model.parameters())         # Har bir parametr tensoridagi sonlar miqdorini olib, hammasini qo'shamiz

total_params = count_parameters(model)                        # Funksiyani modelimizga qo'llaymiz
print(f"Total parameters (including frozen 4-bit): {total_params:,}")   # Natijani mingliklarni vergul bilan ajratib chiqaramiz → 615,606,272

Total parameters (including frozen 4-bit): 615,606,272


# dataset library | load_dataset



*  instruction tuning




In [7]:

dataset = load_dataset("yahma/alpaca-cleaned", split="train")   # "alpaca-cleaned" datasetini internetdan yuklaymiz, faqat "train" qismini
dataset
dataset[1]                                                     # Dataset haqida qisqa ma'lumotni ko'rsatamiz

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

In [8]:
def generate_prompt(example):                     # Bitta misoldan (instruction, input, output) bitta prompt matni yasaydigan funksiya
  instruction = example['instruction']            # Topshiriq / savolni olamiz
  input_text = example['input']                   # Qo'shimcha ma'lumotni olamiz (ko'pincha bo'sh "")
  output_text = example['output']                 # To'g'ri javobni olamiz

  if input_text:                                  # Agar input bo'sh bo'lmasa (qo'shimcha ma'lumot bor bo'lsa)...
    return(                                       # ...3 qismli shablon qaytaramiz:
        "### Instruction:\n"                      # Topshiriq sarlavhasi + yangi qator
        f"{instruction}\n\n"                      # Topshiriq matni + bo'sh qator
        "### Input:\n"                            # Input sarlavhasi
        f"{input_text}\n\n"                       # Input matni + bo'sh qator
        "### Response:\n"                         # Javob sarlavhasi
        f"{output_text}"                          # Javob matni
    )
  else:                                           # Agar input bo'sh bo'lsa...
    return(                                       # ...2 qismli shablon qaytaramiz (Input qismisiz):
       "### Instruction:\n"                       # Topshiriq sarlavhasi
       f"{instruction}\n\n"                       # Topshiriq matni
       "### Response:\n"                          # Javob sarlavhasi
       f"{output_text}"                           # Javob matni
    )

# generate_prompt(dataset[0])                     → tekshirish uchun: birinchi misol qanday ko'rinishini ko'rish

def formatting_func(example):                     # map() uchun yordamchi funksiya
  return {'text': generate_prompt(example)}       # Yangi 'text' ustuniga tayyor prompt matnini yozamiz

dataset = dataset.map(formatting_func)            # Funksiyani datasetdagi HAR BIR qatorga qo'llaymiz → yangi 'text' ustuni qo'shiladi

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [9]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [10]:
dataset = dataset.select(range(7000))     # 51 760 ta misoldan faqat birinchi 7000 tasini qoldiramiz (o'qitish tezroq bo'lishi uchun)


In [11]:
dataset = dataset.shuffle(seed=42)        # Qolgan 7000 ta misolni aralashtiramiz; seed=42 → har safar bir xil tartibda aralashadi

In [12]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [13]:
# Full Fine-tuning  =>                        → modelning BARCHA parametrlarini qayta o'qitish (qimmat, katta xotira)
# Cheap Fine-tuning =>                        → faqat kichik qismini o'qitish (arzon, kam xotira)
# PEFT => Parameter Efficent Fine Tuning      → parametrlardan tejamkor foydalanib fine-tuning qilish usullari (LoRA shu turga kiradi)
# OOM => Out of Memory                        → GPU xotirasi yetmay qolganda chiqadigan xato

In [13]:


lora_config = LoraConfig(                          # LoRA qanday qo'shilishini belgilaymiz
    r=8,   # rank                               → qo'shiladigan kichik matritsalarning "kengligi" (A: 2048×8, B: 8×2048)
    lora_alpha=16,                            # LoRA ta'sirining kuchi (masshtab): ta'sir = alpha / r = 16 / 8 = 2
    lora_dropout=0.05,                        # O'qitishda LoRA neyronlarining 5% ini tasodifan o'chiradi (yodlab olmaslik uchun)
    task_type="CAUSAL_LM",                    # Vazifa turi: keyingi tokenni bashorat qiluvchi matn modeli
    target_modules=["q_proj", "v_proj"],      # LoRA faqat Attention'dagi Query va Value qatlamlariga qo'shiladi
)

In [14]:
model = get_peft_model(model, lora_config) # LoRA sozlamalarini modelga qo'llaymiz: q_proj va v_proj yoniga kichik A, B matritsalar qo'shiladi


In [15]:
model.print_trainable_parameters()            # Nechta parametr o'qitilishini va umumiy sondan necha foiz ekanini chiqaramiz

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [16]:
# QLoRa    → Quantized LoRA: model 4 bitga siqiladi + ustiga LoRA qo'shiladi (siz hozir shuni qilyapsiz)
# LoRa     → Low-Rank Adaptation: asl model muzlatiladi, yoniga kichik A va B matritsalar qo'shilib, faqat ular o'qitiladi

In [17]:
training_args = TrainingArguments(            # O'qitish jarayonining barcha sozlamalari
    per_device_train_batch_size=1,            # Har bir qadamda GPU'ga 1 ta misol beriladi (xotira kam ketishi uchun)
    gradient_accumulation_steps=4,   # VRAM    → 4 qadam gradient yig'ib, keyin bir marta yangilaydi → amalda batch = 4
    num_train_epochs=2,   # overfit            → dataset 2 marta to'liq ko'rib chiqiladi (ko'p bo'lsa overfit bo'ladi)
    logging_steps=20,                         # Har 20 qadamda loss'ni ekranga chiqaradi
    output_dir="./jaydari_gpt",               # Natijalar (checkpoint) saqlanadigan papka
    save_strategy="epoch",                    # Har bir epoch tugaganda modelni saqlaydi
    bf16=True,
    fp16=False,                                # 16 bitli hisoblash → tezroq va kam xotira
    report_to="none"                          # Natijalarni tashqi xizmatlarga (wandb va h.k.) yubormaydi
)

In [ ]:
# SFTTrainer                                      → Supervised Fine-Tuning uchun tayyor "o'qituvchi" klass
trainer = SFTTrainer(                             # O'qitishni boshqaruvchi obyekt yaratamiz
    model=model,                                  # O'qitiladigan model (4 bit + LoRA qo'shilgan)
    train_dataset=dataset,                        # O'qitish ma'lumoti (7000 ta misol)
    formatting_func=lambda x: x['text'],          # Har bir misoldan qaysi matnni olishni aytamiz → tayyor 'text' ustuni
    args=training_args                            # Oldingi hujayradagi sozlamalar (batch, epoch, fp16...)
)


trainer.train()                                   # O'qitishni boshlaymiz (3500 qadam, har 20 qadamda loss chiqadi)
model.save_pretrained('jaydari_gpt')              # O'qitilgan LoRA og'irliklarini 'jaydari_gpt' papkasiga saqlaymiz
tokenizer.save_pretrained('jaydari_gpt')          # Tokenizerni ham shu papkaga saqlaymiz (model bilan birga ishlatish uchun)

Adding EOS to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Step,Training Loss
20,1.525155
40,1.579154
60,1.319199
80,1.371334
100,1.304298
120,1.323936
140,1.348747
160,1.244671
180,1.308055
200,1.215278


Step,Training Loss
20,1.525155
40,1.579154
60,1.319199
80,1.371334
100,1.304298
120,1.323936
140,1.348747
160,1.244671
180,1.308055
200,1.215278


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM   # Tokenizer va matn yozuvchi modelni yuklovchi klasslar
from peft import PeftModel                                     # Saqlangan LoRA og'irliklarini asl modelga ulash uchun klass
import torch                                                   # PyTorch (dtype va GPU uchun)

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"           # Asl (fine-tuning qilinmagan) modelning nomi

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)  # Asl modelning tokenizerini yuklaymiz

model_base = AutoModelForCausalLM.from_pretrained(             # Asl modelni yuklaymiz (bu safar quantization'siz)
    base_model_id,                                             # Qaysi model
    device_map="auto",                                         # GPU bo'lsa GPU'ga, bo'lmasa CPU'ga avtomatik joylaydi
    torch_dtype=torch.bfloat16                                 # Og'irliklarni 16 bitli bfloat16 formatda yuklaydi (~2.2 GB)
)

model_base.eval()                                              # Bashorat rejimi (Dropout o'chadi, o'qitish yo'q)

In [ ]:
prompt = "Explain what machine learning is in simple words."     # Asl modelga beriladigan savol

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)   # Savolni tensorga aylantirib, model turgan qurilmaga (GPU/CPU) ko'chiramiz

with torch.no_grad():                         # Gradient hisoblamaymiz (faqat javob olyapmiz)
    output_base = model_base.generate(        # Asl model javobni token-token yozadi
        **inputs_base,                        # input_ids va attention_mask ni beramiz
        max_new_tokens=120,                   # Ko'pi bilan 120 ta yangi token yozsin
        temperature=0.7,                      # Tasodifiylik darajasi (o'rtacha)
        do_sample=True                        # Ehtimollik bo'yicha tasodifiy tanlaydi (har safar javob biroz boshqacha)
    )

print("===== BASE MODEL OUTPUT =====")         # Sarlavha: bu ASL modelning javobi
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))   # Javob ID'larini matnga aylantirib chiqaramiz

# FINE-TUNED MODEL INFERENCE

In [ ]:
model_path = "jaydari_gpt"                            # O'qitilgan LoRA va tokenizer saqlangan papka

tokenizer_ft = AutoTokenizer.from_pretrained(model_path)   # Tokenizerni o'sha papkadan yuklaymiz (saqlagan tokenizer)

model_ft = PeftModel.from_pretrained(                 # Asl model + o'qitilgan LoRA = fine-tuned model
    model_base,                                       # Asosiy model (oldin yuklangan asl TinyLlama)
    model_path                                        # LoRA og'irliklari turgan papka (adapter_model.safetensors)
)

model_ft.eval()                                       # Bashorat rejimi (LoRA dropout o'chadi)

In [ ]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""                                           # O'qitishdagi Alpaca formatidagi prompt: topshiriq + bo'sh "Response" qismi

inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)   # Promptni tensorga aylantirib, model turgan qurilmaga ko'chiramiz

with torch.no_grad():                         # Gradient hisoblamaymiz (faqat javob olyapmiz)
    output_ft = model_ft.generate(            # Fine-tuned model javobni token-token yozadi
        **inputs_ft,                          # input_ids va attention_mask ni beramiz
        max_new_tokens=120,                   # Ko'pi bilan 120 ta yangi token
        temperature=0.7,                      # Tasodifiylik darajasi (o'rtacha)
        do_sample=True                        # Ehtimollik bo'yicha tasodifiy tanlaydi
    )

print("===== FINE-TUNED MODEL OUTPUT =====")   # Sarlavha: bu O'QITILGAN modelning javobi
print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))   # Javob ID'larini matnga aylantirib chiqaramiz

In [ ]:
!zip -r jaydari_gpt jaydari_gpt               # 'jaydari_gpt' papkasini ichidagi hamma narsa bilan 'jaydari_gpt.zip' arxiviga siqamiz


In [ ]:
from google.colab import files                # Colab'dan kompyuterga fayl yuklab olish moduli
files.download("jaydari_gpt.zip")             # Zip faylni brauzer orqali kompyuteringizga yuklab olamiz


In [ ]:
############################
#      MODEL TESTING       #                  → keyingi bo'lim: modelni sinash
############################

In [ ]:
!pip install -q langdetect                    # Matn qaysi tilda yozilganini aniqlaydigan kutubxonani o'rnatamiz (-q → kam xabar chiqaradi)


In [ ]:
!unzip jaydari_gpt.zip     # Yuklangan zip arxivni ochamiz → 'jaydari_gpt' papkasi qayta paydo bo'ladi

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM   # Tokenizer va matn yozuvchi modelni yuklovchi klasslar
from peft import PeftModel, PeftConfig                         # PeftModel → LoRA'ni ulash; PeftConfig → LoRA sozlamalarini o'qish
import torch                                                   # PyTorch (dtype va GPU uchun)

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"           # Asl modelning nomi (Hugging Face'dan yuklanadi)
adapter_dir = "/content/jaydari_gpt"                           # Unzip qilingan LoRA papkasining to'liq yo'li

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)         # Tokenizerni LoRA papkasidan yuklaymiz (biz saqlagan tokenizer)
tokenizer.pad_token = tokenizer.eos_token                      # Padding uchun </s> tokenidan foydalanamiz (bir nechta gapni batch qilganda kerak)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(             # Asl TinyLlama modelini yuklaymiz
    base_model_id,                                             # Qaysi model
    device_map="auto",                                         # GPU bo'lsa GPU'ga, bo'lmasa CPU'ga joylaydi
    torch_dtype=torch.bfloat16                                 # 16 bitli formatda yuklaymiz (~2.2 GB)
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)     # Asl model ustiga o'qitilgan LoRA (adapter) ni ulaymiz → fine-tuned model
model.eval()                                                   # Bashorat rejimi (dropout o'chadi)

In [ ]:
from langdetect import detect_langs                       # Matn tilini va ehtimolligini aniqlaydigan funksiya

def ask_jaydari_gpt(prompt: str, max_new_tokens=256):     # Savolni oladi, fine-tuned modeldan javob qaytaradi; javob uzunligi standart 256 token
    try:                                                  # Tilni aniqlashga harakat qilamiz...
        langs = detect_langs(prompt)                      # Ehtimolli tillar ro'yxati → masalan [en:0.99, de:0.01]
        lang_code = langs[0].lang                         # Eng ehtimolli til kodi → 'en'
        confidence = langs[0].prob                        # Uning ehtimolligi → 0.99
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")   # Natijani chiqaramiz
    except:                                               # Til aniqlanmasa (masalan, faqat raqam yoki emoji)...
        lang_code = "unknown"                             # ...til noma'lum deb belgilaymiz
        confidence = 0.0                                  # ...ishonch 0

    # Block only if we're confident it's NOT English
    if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:   # Ingliz tili emas + 95% ishonch + matn 20 belgidan uzun bo'lsa...
        return "Sorry, you should ask a question only in English."     # ...javob bermasdan rad etamiz

    # Proceed with generation
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"   # Savolni o'qitishdagi Alpaca formatiga joylaymiz (strip → chetdagi bo'sh joylarni olib tashlaydi)
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")      # Promptni tensorga aylantirib, GPU'ga ko'chiramiz

    outputs = model.generate(                             # Model javobni token-token yozadi
        **inputs,                                         # input_ids va attention_mask
        max_new_tokens=max_new_tokens,                    # Funksiya argumentidan keladigan uzunlik chegarasi
        temperature=0.7,                                  # Tasodifiylik darajasi
        top_p=0.9,                                        # Faqat eng ehtimolli tokenlardan (jami 90% ehtimollikgacha) tanlaydi
        do_sample=True,                                   # Ehtimollik bo'yicha tasodifiy tanlash yoqilgan
        eos_token_id=tokenizer.eos_token_id               # </s> chiqsa, yozishni to'xtatadi
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)   # Butun natijani (prompt + javob) matnga aylantiramiz
    return decoded[len(full_prompt):].strip()             # Prompt uzunligicha belgini kesib tashlab, faqat javob qismini qaytaramiz

In [ ]:
ask_jaydari_gpt('Солнце светит ярко сегодня.')